# Atlantic Haven Hotels — Prédiction d'annulation de réservation
**Examen final Machine Learning & Data Science — M1 (ISPM Madagascar)**

Objectif : prédire `reservation_annulee` (1 = annulée, 0 = maintenue).
Métrique principale : **F1-score de la classe 1 (annulation)**.

Contrainte clé : les données sont **chronologiques** et le test est postérieur au train.
On utilise donc une **validation temporelle** (jamais de mélange aléatoire passé/futur).

Sommaire : 1. Imports · 2. Config/graines · 3. Chargement · 4. EDA · 5. Préparation ·
6. Validation temporelle · 7. Baseline LogReg · 8. Feature engineering ·
9. Comparaison des modèles · 10. Optimisation · 11. Optimisation du seuil ·
12. Analyse des erreurs · 13. Interprétation · 14. Modèle final · 15. submission.csv · 16. Vérifications.

## 1. Imports

In [ ]:
import time, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             confusion_matrix, roc_auc_score, classification_report)

## 2. Configuration et graines aléatoires
On fixe la graine pour la reproductibilité.

In [ ]:
RS = 42
np.random.seed(RS)
DATA = "data"   # sous-dossier contenant les CSV (architecture rangée)
TARGET, ID = "reservation_annulee", "reservation_id"

## 3. Chargement des données

In [ ]:
train = pd.read_csv(f"{DATA}/reservations_train.csv")
test  = pd.read_csv(f"{DATA}/reservations_test.csv")
for c in ["date_reservation", "date_arrivee"]:
    train[c] = pd.to_datetime(train[c]); test[c] = pd.to_datetime(test[c])
print("train:", train.shape, "| test:", test.shape)
print("colonne absente du test:", set(train.columns) - set(test.columns))
train.head(3)

## 4. Analyse exploratoire (EDA)
On vérifie dimensions, types, valeurs manquantes, doublons, distribution de la cible
et surtout l'**ordre temporel** train vs test.

In [ ]:
print("Types:\n", train.dtypes.value_counts())
print("\nValeurs manquantes (train):")
print(train.isna().sum()[lambda s: s > 0])
print("\nDoublons:", train.duplicated().sum(),
      "| IDs uniques:", train[ID].is_unique,
      "| IDs communs train/test:", len(set(train[ID]) & set(test[ID])))
print("\nDistribution cible:")
print(train[TARGET].value_counts(normalize=True).round(3))

In [ ]:
# Ordre temporel : le test est-il postérieur au train ?
print("train date_reservation:", train.date_reservation.min().date(), "->", train.date_reservation.max().date())
print("test  date_reservation:", test.date_reservation.min().date(),  "->", test.date_reservation.max().date())
print("train trié chronologiquement:", train.date_reservation.is_monotonic_increasing)
print("test  trié chronologiquement:", test.date_reservation.is_monotonic_increasing)

In [ ]:
# Taux d'annulation selon quelques variables clés
for c in ["tarif_remboursable", "type_acompte", "canal_reservation"]:
    print(f"\n-- {c} --")
    print(train.groupby(c)[TARGET].mean().round(3))
train["is_direct"] = train["agent_id"].isna()
print("\nRéservation directe vs via agent:")
print(train.groupby("is_direct")[TARGET].mean().round(3))

In [ ]:
# Graphiques EDA
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
train[TARGET].value_counts().sort_index().plot.bar(ax=ax[0], color=["#4C78A8","#E45756"])
ax[0].set_title("Distribution de la cible")
train.groupby("type_acompte")[TARGET].mean().sort_values().plot.bar(ax=ax[1], color="#F58518")
ax[1].set_title("Taux annulation / type_acompte")
ax[2].hist(train[train[TARGET]==0].delai_reservation_jours, bins=40, alpha=.6, density=True, label="maintenue", color="#4C78A8")
ax[2].hist(train[train[TARGET]==1].delai_reservation_jours, bins=40, alpha=.6, density=True, label="annulée", color="#E45756")
ax[2].set_xlim(0,200); ax[2].legend(); ax[2].set_title("Délai de réservation par classe")
plt.tight_layout(); plt.show()

**Constats.** 8000 lignes (train) / 2000 (test), 25,8 % d'annulations (classe déséquilibrée →
le F1 de la classe 1 est justifié, l'accuracy serait trompeuse). Aucun doublon. Cinq colonnes ont des
manquants ; `agent_id` manquant signifie une **réservation directe** (information métier). Le train
s'arrête au 2025-05-24 et le test commence exactement après : **le test est le futur du train**, ce qui
impose une validation temporelle. Signaux forts : acompte total → peu d'annulations, tarif remboursable
et long délai → plus d'annulations.

## 5. Préparation des données & Feature engineering
Toutes les nouvelles variables sont **calculables au moment de la réservation** (pas de fuite du futur ni
de la cible). L'historique client (`taux_annul_client`) utilise uniquement le passé propre du client.
On exclut `reservation_id` (identifiant), les dates brutes, `hotel_id` (84 modalités, redondant) et
`agent_id` (remplacé par un indicateur `reservation_directe`).

In [ ]:
def add_features(df):
    df = df.copy()
    # Composantes temporelles (connues à la réservation)
    df["res_mois"]      = df["date_reservation"].dt.month
    df["res_jour_sem"]  = df["date_reservation"].dt.dayofweek
    df["res_annee"]     = df["date_reservation"].dt.year
    df["res_trimestre"] = df["date_reservation"].dt.quarter
    df["arr_mois"]      = df["date_arrivee"].dt.month
    sais = {12:"hiver",1:"hiver",2:"hiver",3:"printemps",4:"printemps",5:"printemps",
            6:"ete",7:"ete",8:"ete",9:"automne",10:"automne",11:"automne"}
    df["arr_saison"]    = df["arr_mois"].map(sais)
    # Séjour / occupation
    df["total_personnes"]  = df["adultes"] + df["enfants"].fillna(0)
    df["pers_par_chambre"] = df["total_personnes"] / df["chambres"].replace(0, np.nan)
    df["a_enfants"]        = (df["enfants"].fillna(0) > 0).astype(int)
    # Prix / remise
    df["prix_total_par_nuit"] = df["montant_total_eur"] / df["nuits"].replace(0, np.nan)
    df["montant_remise_eur"]  = df["montant_total_eur"] * df["remise_pct"] / 100.0
    df["remise_forte"]        = (df["remise_pct"] >= 10).astype(int)
    # Historique CLIENT (passé propre du client -> pas de fuite de la cible courante)
    df["taux_annul_client"]   = (df["annulations_passees"] / df["reservations_passees"].replace(0, np.nan)).fillna(0)
    df["client_a_historique"] = (df["reservations_passees"] > 0).astype(int)
    # Réservation directe (agent_id manquant)
    df["reservation_directe"] = df["agent_id"].isna().astype(int)
    df["delai_long"]          = (df["delai_reservation_jours"] >= 60).astype(int)
    return df

train_fe = add_features(train.drop(columns=["is_direct"]))
test_fe  = add_features(test)

drop = [ID, TARGET, "date_reservation", "date_arrivee", "agent_id", "hotel_id"]
num_cols = [c for c in train_fe.select_dtypes(include="number").columns if c not in drop]
cat_cols = [c for c in train_fe.columns if c not in drop and c not in num_cols
            and c not in ["date_reservation", "date_arrivee"]]
base_num = [c for c in num_cols if c in train.columns]   # sans FE (pour mesurer l'apport)
base_cat = [c for c in cat_cols if c in train.columns]
print(f"{len(num_cols)} num + {len(cat_cols)} cat après FE")

Le prétraitement (`make_prep`) est défini à la section 6 juste avant son usage. Il est appris **uniquement sur le pli d'entraînement** (via `Pipeline`),
jamais sur le test. `handle_unknown="ignore"` gère les catégories jamais vues ; `SimpleImputer`
impute les manquants avec la médiane (numérique) / le mode (catégoriel) appris sur le train.

## 6. Validation temporelle
Les données étant triées par date, on garde les **20 % de réservations les plus récentes** comme
validation (holdout chronologique). Une validation croisée aléatoire mélangerait passé et futur et
donnerait une estimation trop optimiste (fuite temporelle). Ici on simule la réalité : entraîner sur
l'ancien, prédire le récent.

In [ ]:
# Prétraitement (défini ici pour être toujours disponible avant eval_model)
def make_prep(numc, catc):
    num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc",  StandardScaler())])
    cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                         ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=10))])
    return ColumnTransformer([("num", num_pipe, numc), ("cat", cat_pipe, catc)])

y = train_fe[TARGET].values
n = len(train_fe); cut = int(n * 0.80)
print(f"train[0:{cut}] (jusqu'au {train_fe.date_reservation.iloc[cut-1].date()}) "
      f"| valid[{cut}:{n}] ({train_fe.date_reservation.iloc[cut].date()} -> {train_fe.date_reservation.iloc[-1].date()})")

def split(numc, catc):
    X = train_fe[numc + catc]
    return X.iloc[:cut], y[:cut], X.iloc[cut:], y[cut:]

def eval_model(model, numc, catc, name, thr=0.5):
    Xtr, ytr, Xva, yva = split(numc, catc)
    pipe = Pipeline([("prep", make_prep(numc, catc)), ("clf", model)])
    t0 = time.time(); pipe.fit(Xtr, ytr); dt = time.time() - t0
    p = pipe.predict_proba(Xva)[:, 1]; pred = (p >= thr).astype(int)
    return (dict(model=name, f1=f1_score(yva, pred), precision=precision_score(yva, pred),
                 recall=recall_score(yva, pred), roc_auc=roc_auc_score(yva, p),
                 train_time_s=round(dt, 2)), pipe, p, yva)

## 7. Baseline obligatoire — Régression logistique
Premier modèle de référence (features de base, sans feature engineering).

In [ ]:
res, _, p_bl, yva = eval_model(
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RS),
    base_num, base_cat, "LogReg baseline")
print({k: (round(v,4) if isinstance(v,float) else v) for k,v in res.items()})
print("\nMatrice de confusion (seuil 0.5):\n", confusion_matrix(yva, (p_bl>=0.5).astype(int)))
print("\n", classification_report(yva, (p_bl>=0.5).astype(int), digits=3))

La baseline atteint **F1 ≈ 0.45** (classe 1). L'accuracy seule serait trompeuse : prédire
toujours « 0 » donnerait ~74 % d'accuracy mais un rappel nul. On regarde donc précision, rappel et F1.

## 8–9. Feature engineering & comparaison des modèles
On compare, sur **le même holdout temporel**, régression logistique / Random Forest / HistGradientBoosting,
sans puis avec feature engineering (au seuil 0.5).

In [ ]:
rows = []
for tag, (nc, cc) in {"(sans FE)": (base_num, base_cat), "+ FE": (num_cols, cat_cols)}.items():
    rows.append(eval_model(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RS), nc, cc, f"LogReg {tag}")[0])
    rows.append(eval_model(RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RS, n_jobs=-1), nc, cc, f"RandomForest {tag}")[0])
    rows.append(eval_model(HistGradientBoostingClassifier(random_state=RS, class_weight="balanced"), nc, cc, f"HistGB {tag}")[0])
comp = pd.DataFrame(rows)[["model","f1","precision","recall","roc_auc","train_time_s"]].round(4)
comp

**Lecture.** Les Random Forest s'effondrent au seuil 0.5 (rappel ~3 %) : leurs probabilités
sont mal calibrées → ce n'est pas qu'elles sont mauvaises, c'est que 0.5 est un mauvais seuil pour elles
(d'où l'étape 11). La **régression logistique a le meilleur ROC-AUC (~0.66)** et le meilleur F1, et le
feature engineering l'améliore légèrement. Modèle linéaire = plus robuste au décalage temporel,
interprétable et rapide → on le retient comme modèle final.

## 10. Optimisation des hyperparamètres (modèle retenu : LogReg)
Recherche de la régularisation `C` par **TimeSeriesSplit** (4 plis) sur la portion d'entraînement
uniquement, avec le score `average_precision` (PR-AUC, centré sur la classe 1, indépendant du seuil).
La validation temporelle n'est jamais touchée pendant le tuning.

In [ ]:
Xtr, ytr, Xva, yva = split(num_cols, cat_cols)
tscv = TimeSeriesSplit(n_splits=4)
grid = GridSearchCV(
    Pipeline([("prep", make_prep(num_cols, cat_cols)),
              ("clf", LogisticRegression(max_iter=3000, random_state=RS))]),
    param_grid={"clf__C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]},
    scoring="average_precision", cv=tscv, n_jobs=-1)
grid.fit(Xtr, ytr)
best_C = grid.best_params_["clf__C"]
print("Meilleur C:", best_C, "| PR-AUC CV = %.4f" % grid.best_score_)

final_model = LogisticRegression(C=best_C, max_iter=3000, random_state=RS)
res, pipe_final, proba_val, yva = eval_model(final_model, num_cols, cat_cols, f"LogReg tuné (C={best_C})")
print("Modèle tuné @0.5:", {k:(round(v,4) if isinstance(v,float) else v) for k,v in res.items()})

## 11. Optimisation du seuil de décision
Le seuil 0.5 n'est pas optimal. On cherche le seuil qui **maximise le F1 de la classe 1** en utilisant
uniquement les probabilités de **validation**.

In [ ]:
ths = np.linspace(0.05, 0.95, 181)
f1s = [f1_score(yva, (proba_val >= t).astype(int)) for t in ths]
best_thr = float(ths[int(np.argmax(f1s))]); best_f1 = float(max(f1s))
print(f"Seuil optimal = {best_thr:.2f}  ->  F1 = {best_f1:.4f}")

pred_best = (proba_val >= best_thr).astype(int)
print("Précision=%.3f  Rappel=%.3f" % (precision_score(yva, pred_best), recall_score(yva, pred_best)))
print("Confusion:\n", confusion_matrix(yva, pred_best))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(ths, f1s, color="#4C78A8"); ax[0].axvline(best_thr, ls="--", color="#E45756")
ax[0].axvline(0.5, ls=":", color="grey"); ax[0].set_xlabel("seuil"); ax[0].set_ylabel("F1 (classe 1)")
ax[0].set_title(f"F1 vs seuil (optimum={best_thr:.2f})")
cm = confusion_matrix(yva, pred_best); ax[1].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2): ax[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
ax[1].set_xticks([0,1]); ax[1].set_xticklabels(["prévu 0","prévu 1"])
ax[1].set_yticks([0,1]); ax[1].set_yticklabels(["réel 0","réel 1"]); ax[1].set_title("Confusion (seuil optimal)")
plt.tight_layout(); plt.show()

**Compromis.** Abaisser le seuil augmente le **rappel** (on rate moins d'annulations, moins de
faux négatifs) mais fait baisser la **précision** (plus de fausses alertes). Le seuil qui maximise le F1
privilégie le rappel car, sur ces données, la précision atteignable reste modérée.

## 12. Analyse des erreurs

In [ ]:
val = train_fe.iloc[cut:].copy()
val["proba"] = proba_val; val["pred"] = pred_best; val["reel"] = yva
val["cat_err"] = np.select(
    [(val.reel==1)&(val.pred==1), (val.reel==0)&(val.pred==0),
     (val.reel==0)&(val.pred==1), (val.reel==1)&(val.pred==0)],
    ["VP","VN","FP","FN"], default="NA")
profil = val.groupby("cat_err").agg(
    n=("proba","size"), delai=("delai_reservation_jours","mean"),
    prix=("montant_total_eur","mean"),
    remboursable=("tarif_remboursable", lambda s:(s=="oui").mean()),
    acompte_aucun=("type_acompte", lambda s:(s=="aucun").mean())).round(2)
print(profil)
cols = ["reservation_id","tarif_remboursable","type_acompte","delai_reservation_jours","canal_reservation","montant_total_eur","proba"]
print("\n5 faux positifs (proba la plus haute):"); display(val[val.cat_err=="FP"].nlargest(5,"proba")[cols])
print("5 faux négatifs (proba la plus basse):");   display(val[val.cat_err=="FN"].nsmallest(5,"proba")[cols])

In [ ]:
# Performance par région (robustesse par sous-groupe)
by_reg = val.groupby("region_hotel").apply(
    lambda g: pd.Series({"n":len(g), "f1": f1_score(g.reel, g.pred) if g.reel.sum()>0 else np.nan})).round(3)
print(by_reg.sort_values("f1"))

**Constats.** Les **faux négatifs** (annulations ratées) ressemblent à de « bons » dossiers :
acompte payé, tarif non remboursable, délai court — ils paraissent sûrs mais annulent quand même.
Les **faux positifs** (fausses alertes) cumulent tarif remboursable et absence d'acompte : ils
*paraissent* risqués mais sont honorés. Le F1 varie de ~0.40 à ~0.57 selon la région ; les petits
sous-groupes sont à interpréter avec prudence.

## 13. Interprétation
Permutation importance (impact réel sur le F1) + coefficients signés de la régression logistique.

In [ ]:
perm = permutation_importance(pipe_final, Xva, yva, scoring="f1", n_repeats=10, random_state=RS, n_jobs=-1)
imp = pd.DataFrame({"feature": Xva.columns, "importance": perm.importances_mean}).sort_values("importance", ascending=False)
print(imp.head(12).to_string(index=False))

feat_names = pipe_final.named_steps["prep"].get_feature_names_out()
coefs = pd.DataFrame({"feature": feat_names, "coef": pipe_final.named_steps["clf"].coef_[0]})
top = pd.concat([coefs.nlargest(8,"coef"), coefs.nsmallest(8,"coef")]).sort_values("coef")

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ii = imp.head(12).iloc[::-1]; ax[0].barh(ii.feature, ii.importance, color="#54A24B")
ax[0].set_title("Permutation importance (perte de F1)")
ax[1].barh(top.feature, top.coef, color=np.where(top.coef>0,"#E45756","#4C78A8"))
ax[1].set_title("Coefficients LogReg (+ = augmente l'annulation)")
plt.tight_layout(); plt.show()

**Variables les plus influentes.** Le **délai de réservation** (long → risque ↑), le
**type d'acompte** (aucun → risque ↑ ; total → risque ↓↓), le **tarif remboursable** (→ risque ↑),
le **taux d'annulation passé du client**, et la **réservation directe** (→ risque ↓). Ces effets sont
cohérents avec l'intuition métier : un client qui n'a rien versé et peut annuler sans frais, longtemps
à l'avance, est le plus volatil.

## 14. Modèle final — réentraînement sur tout le train
Une fois features, modèle (LogReg C optimal) et seuil fixés, on réentraîne sur **l'intégralité** des
8000 réservations, puis on prédit le test.

In [ ]:
Xall = train_fe[num_cols + cat_cols]
pipe_all = Pipeline([("prep", make_prep(num_cols, cat_cols)),
                     ("clf", LogisticRegression(C=best_C, max_iter=3000, random_state=RS))])
pipe_all.fit(Xall, y)
proba_test = pipe_all.predict_proba(test_fe[num_cols + cat_cols])[:, 1]
pred_test  = (proba_test >= best_thr).astype(int)
print("Prédictions test — taux d'annulation prédit:", round(pred_test.mean(), 3))

## 15. Génération de `submission.csv`
Exactement 3 colonnes, ordre des IDs identique au test.

In [ ]:
submission = pd.DataFrame({
    ID: test_fe[ID].values,
    "probabilite_annulation": np.round(proba_test, 6),
    "reservation_annulee": pred_test})
submission.to_csv("submission.csv", index=False)
submission.head()

## 16. Vérifications finales

In [ ]:
sample = pd.read_csv(f"{DATA}/sample_submission.csv")
assert len(submission) == 2000, "doit avoir 2000 lignes"
assert list(submission.columns) == ["reservation_id","probabilite_annulation","reservation_annulee"]
assert submission[ID].isna().sum() == 0
assert (submission[ID].values == test_fe[ID].values).all(), "ordre != test"
assert (submission[ID].values == sample[ID].values).all(), "ordre != sample_submission"
assert submission.probabilite_annulation.between(0,1).all()
assert sorted(submission.reservation_annulee.unique()) == [0,1]
print("Toutes les vérifications passent.")
print(submission.describe(include='all'))